In [1]:
# Install required packages.
# Note: Using `%pip` and `-U` ensures proper upgrades in the active Colab kernel.
%pip install -U langchain "langchain[google-genai]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 9.5 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.0
    Uninstalling langchain-core-1.6.0:
      Successfully uninstalled langchain-core-1.6.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-

------------------------------
## Method C (Production Standard): Dynamic Profiles via Middleware

While Methods A and B are great for static configurations, production applications often require the **same agent instance** to behave differently based on the user or the specific task (e.g., a "creative" profile for brainstorming vs. a "strict" profile for data extraction).

Rebuilding the agent or model object for every request is inefficient and violates the **Open/Closed Principle** (SOLID).

### The Solution: Runtime Context & Middleware
LangChain's modern architecture allows us to intercept the execution flow. We can pass a `context` dictionary during `.invoke()`, and a custom **middleware** will read it and dynamically bind the hyperparameters to the model *just milliseconds before inference*, without altering the base agent in memory.

### Key Benefits:
1. **Memory Efficiency**: The agent is instantiated only once.
2. **Dynamic Adaptation**: Change temperature, tokens, or penalties per request.
3. **Provider Abstraction**: The middleware can map generic profile names to provider-specific parameters (e.g., `max_output_tokens` for Gemini).

In [7]:
import os
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware # Importamos la clase base oficial
from langchain.chat_models import init_chat_model
from google.colab import userdata

# 1. Setup API Key
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

# 2. Define the Middleware as a CLASS (The "Profile Manager")
class DynamicProfileMiddleware(AgentMiddleware):
    """
    Middleware that intercepts the runtime context to inject hyperparameter profiles
    directly into the model call at runtime.
    """
    def wrap_model_call(self, runtime, handler):
        """
        Intercepts the model call. We read the context and bind parameters
        before the handler executes the actual LLM call.
        """
        # Retrieve dynamic context sent in .invoke()
        context = getattr(runtime, 'context', {}) or {}
        profile_options = context.get("generation_config", {})

        # If the user specified parameters, bind them to the model on the fly
        if profile_options:
            # .bind() creates a temporary copy of the model with these specific parameters
            runtime.model = runtime.model.bind(**profile_options)

        # Continue the execution chain
        return handler(runtime)

# 3. Initialize the Agent ONCE with the middleware registered
# We use init_chat_model for maximum flexibility
base_model = init_chat_model("google_genai:gemini-3.6-flash")

agent = create_agent(
    model=base_model,
    system_prompt="You are a versatile AI assistant. Adapt your style to the requested profile.",
    middleware=[DynamicProfileMiddleware()] # IMPORTANT: Pass an INSTANCE of the class ()
)

# 4. EXECUTION: PROFILE 1 (Creative)
creative_profile = {"temperature": 0.9, "max_output_tokens": 100}

print(" Running with Creative Profile (Temperature 0.9)...")
creative_response = agent.invoke(
    {"messages": [{"role": "user", "content": "Write a 1-sentence sci-fi plot twist."}]},
    context={"generation_config": creative_profile} # Injecting the profile
)
print(f"AI 🤖: {creative_response['messages'][-1].content[0].get('text', '')}\n")

# 5. EXECUTION: PROFILE 2 (Precise/Strict)
precise_profile = {"temperature": 0.1, "max_output_tokens": 50}

print(" Running with Precise Profile (Temperature 0.1)...")
precise_response = agent.invoke(
    {"messages": [{"role": "user", "content": "Write a 1-sentence sci-fi plot twist."}]},
    context={"generation_config": precise_profile} # Injecting a different profile
)
print(f"AI 🤖: {precise_response['messages'][-1].content[0].get('text', '')}")

 Running with Creative Profile (Temperature 0.9)...
AI 🤖: When the human rebels finally breached the alien hive-mind, they discovered Earth was never a conquered colony, but an ancient quarantine zone built to protect the rest of the universe from us.

 Running with Precise Profile (Temperature 0.1)...
AI 🤖: As I successfully deactivated the rogue AI that slaughtered the crew, the terminal screen flickered with my own system diagnostics, revealing I was just a subroutine designed to keep the empty ship flying.
